# IAM — 02: Self-Service Permission Introspection

**Persona:** Geo-Scientist (regular authenticated user)

**Goal:** Discover your own access without administrator intervention:
- List all catalogs you can access
- Inspect effective capabilities on a specific catalog
- Retrieve your assigned roles (catalog-scoped and global)
- Discover which additional roles are available to you

All operations use the scientist's own token — no sysadmin or admin escalation required.

**Env vars required:** `DYNASTORE_BASE_URL`, `DYNASTORE_TOKEN`, `CATALOG_ID`

In [ ]:
import os
import json

import httpx
from dotenv import load_dotenv

load_dotenv()

BASE_URL   = os.environ.get("DYNASTORE_BASE_URL", "http://localhost:8080")
TOKEN      = os.environ.get("DYNASTORE_TOKEN", "")
CATALOG_ID = os.environ.get("CATALOG_ID", "demo-catalog")

client = httpx.Client(
    base_url=BASE_URL,
    headers={
        "Authorization": f"Bearer {TOKEN}",
        "Content-Type": "application/json",
    },
    timeout=30.0,
)
print(f"Connected to {BASE_URL}")
print(f"Catalog ID : {CATALOG_ID}")

In [ ]:
# List all catalogs accessible to the current identity
r = client.get("/me/catalogs")
print(r.status_code, r.text[:400])
assert r.status_code == 200, f"Expected 200, got {r.status_code}: {r.text}"

catalogs_body = r.json()
catalog_list = catalogs_body if isinstance(catalogs_body, list) else catalogs_body.get("catalogs", catalogs_body.get("items", []))

print(f"Accessible catalogs ({len(catalog_list)}):")
for c in catalog_list:
    cid = c.get("id") or c.get("catalog_id") or c
    print(f"  - {cid}")

In [ ]:
# Retrieve the effective authorization (capabilities) for a specific catalog
# The EffectiveAuthorizationResponse aggregates all capabilities derived from
# every role held by this identity on this catalog, including inherited capabilities.
r = client.get(f"/me/catalogs/{CATALOG_ID}")
print(r.status_code, r.text[:500])
assert r.status_code == 200, f"Expected 200, got {r.status_code}: {r.text}"

auth_info = r.json()
capabilities = auth_info.get("capabilities", auth_info.get("permissions", []))
print(f"\nEffective capabilities on catalog '{CATALOG_ID}':")
print(json.dumps(auth_info, indent=2))

In [ ]:
# Retrieve the raw roles assigned to the current identity on this catalog
r = client.get(f"/me/roles/catalogs/{CATALOG_ID}")
print(r.status_code, r.text[:400])
assert r.status_code == 200, f"Expected 200, got {r.status_code}: {r.text}"

roles_body = r.json()
role_names = roles_body if isinstance(roles_body, list) else roles_body.get("roles", [])
print(f"\nCatalog-scoped roles on '{CATALOG_ID}': {role_names}")

In [ ]:
# Retrieve global (platform-level) roles for the current identity
r = client.get("/me/roles/global")
print(r.status_code, r.text[:400])
assert r.status_code == 200, f"Expected 200, got {r.status_code}: {r.text}"

global_body = r.json()
global_roles = global_body if isinstance(global_body, list) else global_body.get("roles", [])
print(f"Global roles: {global_roles}")

In [ ]:
# Discover which roles are available (grantable) on this catalog
r = client.get("/me/available-roles", params={"catalog_id": CATALOG_ID})
print(r.status_code, r.text[:400])
assert r.status_code == 200, f"Expected 200, got {r.status_code}: {r.text}"

available_body = r.json()
available = available_body if isinstance(available_body, list) else available_body.get("roles", [])
role_names_available = [
    r_item.get("name") if isinstance(r_item, dict) else r_item
    for r_item in available
]
print(f"Available roles for catalog '{CATALOG_ID}': {role_names_available}")

## Edge cases

### Case A — No catalog access returns empty list, not 403

When the identity holds no role on any catalog, `/me/catalogs` returns an **empty list**
with status 200. The IAM facade never returns 403 on self-service list endpoints — 403
is reserved for admin endpoints the identity is not authorised to call.

### Case B — Effective capabilities are richer than raw role names

The `EffectiveAuthorizationResponse` aggregates inherited capabilities from all held roles.
A scientist with only `authorized` may inherit `read` and `search` capabilities derived
from the role definition — more information than the raw role name alone.

In [ ]:
# Case A: Use a deliberately invalid token to simulate zero-access identity.
# /me/catalogs must return 200 with an empty list (not 403).
zero_access_client = httpx.Client(
    base_url=BASE_URL,
    headers={"Authorization": "Bearer zero-access-token"},
    timeout=30.0,
)

r = zero_access_client.get("/me/catalogs")
print(r.status_code, r.text[:300])
# 200 with empty list = correct; 401 acceptable when token verification fails
# before the IAM layer is reached (depends on middleware order).
assert r.status_code in (200, 401), (
    f"Expected 200 (empty) or 401 for zero-access token, got {r.status_code}: {r.text}"
)
if r.status_code == 200:
    body = r.json()
    items = body if isinstance(body, list) else body.get("catalogs", body.get("items", []))
    assert len(items) == 0, f"Expected empty catalog list for zero-access token, got: {items}"
    print("200 with empty list — correct: no catalogs accessible.")
else:
    print("401 — token rejected at auth middleware before IAM layer (also acceptable).")
zero_access_client.close()

In [ ]:
# Case B: Verify that the EffectiveAuthorizationResponse contains more keys
# than just a bare list of role names (inherited capabilities present).
r = client.get(f"/me/catalogs/{CATALOG_ID}")
assert r.status_code == 200, f"Expected 200, got {r.status_code}: {r.text}"

effective = r.json()

# The effective auth response must carry capability details, not just role names.
# Accept either a 'capabilities' list or a rich dict with multiple keys.
has_capabilities = (
    "capabilities" in effective
    or "permissions" in effective
    or len(effective) > 1
)
assert has_capabilities, (
    f"Expected EffectiveAuthorizationResponse to carry capability details, got: {effective}"
)

# Roles from /me/roles/catalogs for comparison
r2 = client.get(f"/me/roles/catalogs/{CATALOG_ID}")
assert r2.status_code == 200
raw_roles = r2.json()

print("Raw role assignment:")
print(json.dumps(raw_roles, indent=2))
print("\nEffective authorization (richer):")
print(json.dumps(effective, indent=2))
print("\nEffective response is richer than raw role list:", has_capabilities)

client.close()